In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit

SEED = 42
np.random.seed(SEED) # Initiates the NumPy random generator with a constant seed so we get the same results every time we run it

In [2]:
PROJECT_ROOT = Path.cwd().parents[0]  
DATASET_DIR = PROJECT_ROOT / "Dataset"

SOURCE_CSV = DATASET_DIR / "combined_stage1.csv"

# Output files
TRAIN_CSV = DATASET_DIR / "stage1_train.csv"
VAL_CSV = DATASET_DIR / "stage1_val.csv"
TEST_CSV = DATASET_DIR / "stage1_test.csv"
META_JSON = DATASET_DIR / "stage1_split_metadata.json"

In [3]:
df = pd.read_csv(SOURCE_CSV)

required_cols = {"sample_id", "patient_global", "y"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Identify the image path column 
image_col = None
for cand in ["image_path", "image", "image_file", "filepath", "path"]:
    if cand in df.columns:
        image_col = cand
        break
if image_col is None:
    raise ValueError("Could not find an image path column. Add one to combined_stage1.csv")

In [4]:
# Strategy: patient label = max(y) across their samples
patient_df = (
    df.groupby("patient_global", as_index=False)
      .agg(patient_label=("y", "max"))
)

In [5]:
# 70/15/15 split
sss_1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(sss_1.split(patient_df, patient_df["patient_label"]))

train_patients = patient_df.iloc[train_idx]["patient_global"].tolist()
temp_patients = patient_df.iloc[temp_idx]

# Split temp into val/test equally
sss_2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(sss_2.split(temp_patients, temp_patients["patient_label"]))

val_patients = temp_patients.iloc[val_idx]["patient_global"].tolist()
test_patients = temp_patients.iloc[test_idx]["patient_global"].tolist()

In [6]:
train_df = df[df["patient_global"].isin(train_patients)].copy()
val_df = df[df["patient_global"].isin(val_patients)].copy()
test_df = df[df["patient_global"].isin(test_patients)].copy()

In [7]:
keep_cols = ["sample_id", image_col, "y", "patient_global", "dataset_id"]
keep_cols = [c for c in keep_cols if c in df.columns]

train_df[keep_cols].to_csv(TRAIN_CSV, index=False)
val_df[keep_cols].to_csv(VAL_CSV, index=False)
test_df[keep_cols].to_csv(TEST_CSV, index=False)

meta = {
    "seed": SEED,
    "split": {"train": 0.70, "val": 0.15, "test": 0.15},
    "source_csv": str(SOURCE_CSV),
    "image_col": image_col,
    "counts": {
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "test_samples": len(test_df),
        "train_patients": len(train_patients),
        "val_patients": len(val_patients),
        "test_patients": len(test_patients),
    },
}
META_JSON.write_text(json.dumps(meta, indent=2))

409

In [8]:
# 1) No patient overlap
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

# 2) Label distribution
print("Label distribution (train):\n", train_df["y"].value_counts(normalize=True))
print("Label distribution (val):\n", val_df["y"].value_counts(normalize=True))
print("Label distribution (test):\n", test_df["y"].value_counts(normalize=True))

# 3) Image path existence check (first 5 missing)
missing_paths = []
for p in train_df[image_col].tolist() + val_df[image_col].tolist() + test_df[image_col].tolist():
    candidate = (PROJECT_ROOT / p).expanduser()
    if not candidate.exists():
        missing_paths.append(p)
    # if not Path(p).expanduser().exists():
    #     missing_paths.append(p)

if missing_paths:
    print(f"Missing image paths: {len(missing_paths)} (showing up to 5)")
    print(missing_paths[:5])
else:
    print("All image paths found.")

Label distribution (train):
 y
1.0    0.599698
0.0    0.400302
Name: proportion, dtype: float64
Label distribution (val):
 y
1.0    0.624853
0.0    0.375147
Name: proportion, dtype: float64
Label distribution (test):
 y
1.0    0.585575
0.0    0.414425
Name: proportion, dtype: float64
All image paths found.


In [9]:
# To ensure that they exist
print(TRAIN_CSV, TRAIN_CSV.exists())
print(VAL_CSV, VAL_CSV.exists())
print(TEST_CSV, TEST_CSV.exists())

/Users/dimitriskaltsios/University/Individual/Kaltsios-Individual-Project/Dataset/stage1_train.csv True
/Users/dimitriskaltsios/University/Individual/Kaltsios-Individual-Project/Dataset/stage1_val.csv True
/Users/dimitriskaltsios/University/Individual/Kaltsios-Individual-Project/Dataset/stage1_test.csv True
